# 14 — Security, Elicitation, and RBAC

## Elicitation deep dive

Elicitation is the user-confirmation gate that protects secrets: any tool that might return a Key Vault secret, connection string, password, or certificate private key requires explicit user confirmation in interactive MCP clients before it executes. See [`docs/08_security_and_best_practices.md`](../docs/08_security_and_best_practices.md).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from src.mcp_client import connect

async def list_tools_with_annotations():
    async with connect(read_only=True) as client:
        return await client.list_tools()

tools = await list_tools_with_annotations()
example = tools[0]
print(example.name, "->", getattr(example, "annotations", None))

In [ ]:
# Filter for tools that are explicitly read-only, versus ones that could be
# destructive - useful before wiring any tool list into a fully automated loop.
def is_read_only(tool) -> bool:
    annotations = getattr(tool, "annotations", None)
    return bool(annotations and getattr(annotations, "readOnlyHint", False))

read_only_tools = [t for t in tools if is_read_only(t)]
print(f"{len(read_only_tools)} / {len(tools)} tools are annotated read-only")